<a href="https://colab.research.google.com/github/manish-tanejaa/hf_agents_course/blob/main/HF_Unit1_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q huggingface_hub

In [ ]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except ImportError:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
)
MODEL = "openai/gpt-oss-20b"

In [ ]:
for m in client.models.list().data:
    print(m.id)

whisper-large-v3-turbo
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-v1-english
whisper-large-v3
qwen/qwen3.8-27b
openai/gpt-oss-20b
allam-2-7b
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m


In [ ]:
output = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "The capital of France is"},
    ],
    stream=False,
    max_tokens=1024,
)
print(output.choices[0].message.content)

The capital of France is Paris.


In [ ]:
# This system prompt is a bit more complex and actually contains the function description already appended.
# Here we suppose that the textual description of the tools has already been appended.

SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{{
  "action": "get_weather",
  "action_input": {{"location": "New York"}}
}}


ALWAYS use the following format:

Question: the input question you must answer
Thought: you should always think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
... (this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer. """

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
]

print(messages)

[{'role': 'system', 'content': 'Answer the following questions as best you can. You have access to the following tools:\n\nget_weather: Get the current weather in a given location\n\nThe way you use the tools is by specifying a json blob.\nSpecifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).\n\nThe only values that should be in the "action" field are:\nget_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}\nexample use :\n\n{{\n  "action": "get_weather",\n  "action_input": {{"location": "New York"}}\n}}\n\n\nALWAYS use the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about one action to take. Only one action at a time in this format:\nAction:\n\n$JSON_BLOB (inside markdown cell)\n\nObservation: the result of the action. This Observation is unique, complete, and the source of truth.\n... (thi

In [ ]:
MODEL = "qwen/qwen3.8-27b"

output = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    stream=False,
    max_tokens=1000,
)
print(output.choices[0].message.content)

Thought: I need to find out the current weather in London. I will use the get_weather tool by providing the location "London".
Action:
```json
{
  "action": "get_weather",
  "action_input": {
    "location": "London"
  }
}
```


In [ ]:
# The answer was hallucinated by the model. We need to stop to actually execute the function!
output = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    max_tokens=150,
    stop=["Observation:"], # Let's stop before any actual function is called
)

print(output.choices[0].message.content)

Question: What's the weather in London?
Thought: I need to get the current weather in London. To do this, I will use the get_weather tool with the location set to "London".
Action:
```json
{
  "action": "get_weather",
  "action_input": {
    "location": "London"
  }
}
```


In [ ]:
# Dummy function
def get_weather(location):
    return f"the weather in {location} is sunny with low temperatures. \n"

get_weather('London')

'the weather in London is sunny with low temperatures. \n'

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
    {"role": "assistant", "content": output.choices[0].message.content + "Observation:\n" + get_weather('London')},
]

output = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    stream=False,
    max_tokens=200
)

print(output.choices[0].message.content)

Thought: The tool response indicates that it is sunny with low temperatures in London. This answers the original question. I can now provide the final answer.
Final Answer: The weather in London is sunny with low temperatures.
